# 03. Machine Learning — Atlas de Inteligencia Vial Espacio-Temporal

## 1. Planteamiento del estudio

En los notebooks anteriores preparamos los registros de incidentes viales de la SSC en Ciudad de México durante 2024 y exploramos sus características temporales, vehiculares y geográficas. Aqui retomaremos esos datos para investigar patrones que puedan modelarse y compararse fuera de las tablas descriptivas.

Comenzamos estudiando la fatalidad mediante un baseline y una regresión logística. La comparación mostró cierta capacidad discriminativa, pero no una mejora uniforme en la calidad de las probabilidades, debido a limitaciones como la cantidad limitada de datos y el imbalanceo de clases. Decidimos **reorientar el estudio hacia la distribución espacio-temporal de los incidentes y su composición vehicular**. Conservamos el experimento original y sus resultados en el **Anexo A**.

Nos planteamos dos preguntas:

**1) ¿Cómo cambia la cantidad de incidentes registrados entre zonas y horarios?**

**2) ¿Cómo varía la distribución de los grupos vehiculares registrados según las características disponibles?**

Primero prepararemos una base de trabajo común y después examinaremos cómo representar el territorio para evaluar qué modelos, si los hubiera, aportan información adicional. Los resultados que obtengamos determinarán qué podrá incorporarse al Atlas interactivo que planeamos como producto final.

## 2. Preparación de los datos

Utilizaremos los cinco archivos Parquet generados en el notebook 01. Conservamos sus transformaciones y las categorías del análisis exploratorio para trabajar a partir de una misma base.

In [ ]:
# Montamos drive
import os
from pathlib import Path
from google.colab import drive

mount_point = '/content/drive'
if not os.path.ismount(mount_point):
    drive.mount(mount_point)

ruta_proyecto = Path('/content/drive/MyDrive/ProyectosPython/cdmx-road-safety-intelligence')
ruta_processed = ruta_proyecto / 'data/processed'
if not ruta_processed.is_dir():
    raise FileNotFoundError(f'No se encontró la carpeta de datos procesados: {ruta_processed}')


In [ ]:
# Cargamos los archivos con pandas

import pandas as pd

archivos = {
    'hechos': 'hechos_ssc_clean.parquet',
    'vehiculos': 'vehiculos_clean.parquet',
    'edad': 'lesionados_tipo_edad_clean.parquet',
    'persona': 'lesionados_tipo_persona_clean.parquet',
    'sexo': 'lesionados_tipo_sexo_clean.parquet'
}

datos = {nombre: pd.read_parquet(ruta_processed / archivo) for nombre, archivo in archivos.items()}

for nombre, df in datos.items():
    print(f'{nombre}: {df.shape}')


### 2.1. Integración del grupo vehicular

Para mantener una fila por incidente, partimos de `hechos` y añadiremos el grupo vehicular por `folio`. Utilizaremos la misma agrupación de seis categorías empleada en el notebook 02.

In [ ]:
df_ml = datos['hechos'].copy()

# Creamos y añadimos categorias
mapa_grupos_vehiculo = {
    'AUTOMOVIL': 'Transporte de personas',
    'CAMIONETA': 'Transporte de personas',
    'TAXI': 'Transporte de personas',
    'MOTOCICLETA': 'Motocicleta',
    'BICICLETA': 'Micromovilidad',
    'MONOPATIN': 'Micromovilidad',
    'AUTOBUS DE PASAJEROS': 'Transporte colectivo',
    'MICROBUS': 'Transporte colectivo',
    'METROBUS': 'Transporte colectivo',
    'TROLEBUS': 'Transporte colectivo',
    'TREN LIGERO': 'Transporte colectivo',
    'CAMION DE CARGA': 'Carga',
    'SD': 'Sin dato'
}

vehiculos = datos['vehiculos'][['folio', 'tipo_vehiculo']].copy()
vehiculos['tipo_vehiculo_grupo'] = vehiculos['tipo_vehiculo'].map(mapa_grupos_vehiculo)

df_ml = df_ml.merge(
    vehiculos[['folio', 'tipo_vehiculo_grupo']],
    on='folio', how='left', validate='one_to_one'
)

print(f'Dimensiones: {df_ml.shape}')
print(f'Grupos sin asignar: {df_ml["tipo_vehiculo_grupo"].isna().sum()}')


### 2.2. Bloques horarios

Para trabajar con periodos comparables recuperaremos los cuatro bloques de seis horas utilizados en el EDA: madrugada, mañana, tarde y noche. Mantendremos `fecha_evento` para los análisis temporales que se desarrollen después.

In [ ]:
# Agregamos bloques horarios
def clasificar_horario(hora):
    if 0 <= hora < 6:
        return 'Madrugada'
    elif 6 <= hora < 12:
        return 'Mañana'
    elif 12 <= hora < 18:
        return 'Tarde'
    else:
        return 'Noche'

df_ml['bloque_horario'] = df_ml['hora_evento'].map(lambda hora: clasificar_horario(hora.hour))

print(df_ml['bloque_horario'].value_counts())


### 2.3. Base de trabajo para el análisis espacial

Las coordenadas y la alcaldía utilizada ya se encuentran en los datos limpios. Reuniremos estas variables con la fecha, el bloque horario, el grupo vehicular y la clasificación vial en una tabla de trabajo por incidente.

In [ ]:
columnas_atlas = [
    'folio', 'fecha_evento', 'latitud', 'longitud', 'alcaldia_utilizada',
    'bloque_horario', 'tipo_vehiculo_grupo', 'clasificacion_de_la_vialidad'
]
df_atlas = df_ml[columnas_atlas].copy()
print(f'Base inicial del Atlas: {df_atlas.shape[0]:,} incidentes y {df_atlas.shape[1]} columnas')

## 3. Unidades espaciales

Para comparar la cantidad de incidentes entre zonas y periodos necesitaremos identificar a qué unidad espacial pertenece cada registro. En el notebook 02 utilizamos una cuadrícula hexagonal para el análisis geográfico; comenzaremos recuperando sus criterios de construcción y comprobando que sirva para crear una asignación estable por incidente.

---

## Anexo A. Experimento preliminar de fatalidad

Este experimento examinó si el horario, el grupo vehicular y la clasificación vial aportaban información para estimar la probabilidad de fatalidad entre los incidentes registrados. Aquí conservamos su planteamiento, evaluación y resultados.

### A.1. Variable objetivo

Definimos `fatal` a partir de `personas_fallecidas`: vale 1 si el incidente registró al menos una persona fallecida y 0 en otro caso. Con ella distinguimos la presencia de una consecuencia fatal del número de personas fallecidas.

In [ ]:
# Variable objetivo del experimento preliminar
df_ml['fatal'] = (df_ml['personas_fallecidas'] > 0).astype(int)

print(df_ml['fatal'].value_counts().sort_index())
print(f'Proporción de incidentes fatales: {df_ml["fatal"].mean():.2%}')

### A.2. Predictores

Utilizamos tres variables categóricas: `bloque_horario`, `tipo_vehiculo_grupo` y `clasificacion_de_la_vialidad`. Dejamos la ubicación fuera de este primer modelo para disponer de una referencia basada en características temporales, vehiculares y viales.

In [ ]:
# Definimos variables predictoras
predictores = [
    'bloque_horario',
    'tipo_vehiculo_grupo',
    'clasificacion_de_la_vialidad'
]

X = df_ml[predictores].copy()
y = df_ml['fatal'].copy()

display(X.head())
print(f'Predictores: {X.shape[1]}')
print(f'Incidentes: {len(X):,}')


### A.3. Separación temporal

Para evaluar el comportamiento del modelo en meses posteriores a su ajuste, reservamos septiembre–diciembre como prueba final. Dentro del periodo enero–agosto separamos enero–junio para el ajuste y julio–agosto para validar las primeras comparaciones.

In [ ]:
# Definimos corte y creamos la máscara
fecha_corte = pd.Timestamp('2024-09-01')
mask_train = df_ml['fecha_evento'] < fecha_corte

X_train, X_test = X.loc[mask_train], X.loc[~mask_train]
y_train, y_test = y.loc[mask_train], y.loc[~mask_train]

print(f'Entrenamiento: {len(X_train):,} incidentes ({y_train.sum()} fatales)')
print(f'Prueba:        {len(X_test):,} incidentes ({y_test.sum()} fatales)')
print(f'Prevalencia entrenamiento: {y_train.mean():.2%}')
print(f'Prevalencia prueba:        {y_test.mean():.2%}')


In [ ]:
# Conjunto de validación
fecha_validacion = pd.Timestamp('2024-07-01')
mask_ajuste = df_ml.loc[X_train.index, 'fecha_evento'] < fecha_validacion

X_ajuste, X_val = X_train.loc[mask_ajuste], X_train.loc[~mask_ajuste]
y_ajuste, y_val = y_train.loc[mask_ajuste], y_train.loc[~mask_ajuste]

print(f'Ajuste:      {len(X_ajuste):,} incidentes ({y_ajuste.sum()} fatales)')
print(f'Validación: {len(X_val):,} incidentes ({y_val.sum()} fatales)')
print(f'Prevalencia ajuste:      {y_ajuste.mean():.2%}')
print(f'Prevalencia validación: {y_val.mean():.2%}')


### A.4. Baseline probabilístico

Antes de entrenar un modelo con predictores, establecimos una referencia que asigna a todos los incidentes la prevalencia de fatalidad del conjunto de ajuste. Mediremos el error de sus probabilidades con **Brier score** y **log loss** buscando valores menores en ambas métricas.

In [ ]:
# Establecemos metricas y evaluamos
import numpy as np
from sklearn.metrics import brier_score_loss, log_loss

p_base = y_ajuste.mean()
proba_base_val = np.full(len(y_val), p_base)

brier_base = brier_score_loss(y_val, proba_base_val)
logloss_base = log_loss(y_val, proba_base_val, labels=[0, 1])

print(f'Probabilidad constante: {p_base:.4%}')
print(f'Brier score: {brier_base:.6f}')
print(f'Log loss:    {logloss_base:.6f}')


### A.5. Regresión logística

Probamos una regresión logística para obtener probabilidades distintas según las tres características seleccionadas. Al tratarse de variables categóricas, utilizamos One-Hot Encoding dentro de un pipeline. Conservamos la regularización predeterminada, sin sobremuestreo ni ponderaciones de clase.

In [ ]:
from sklearn.compose import ColumnTransformer
from sklearn.preprocessing import OneHotEncoder
from sklearn.pipeline import Pipeline
from sklearn.linear_model import LogisticRegression

preprocesador = ColumnTransformer([
    ('categoricas', OneHotEncoder(handle_unknown='ignore'), predictores)
])

modelo_logistico = Pipeline([
    ('preprocesamiento', preprocesador),
    ('modelo', LogisticRegression(max_iter=1000, random_state=42))
])

modelo_logistico.fit(X_ajuste, y_ajuste)
proba_log_val = modelo_logistico.predict_proba(X_val)[:, 1]

brier_log = brier_score_loss(y_val, proba_log_val)
logloss_log = log_loss(y_val, proba_log_val, labels=[0, 1])

resultados_val = pd.DataFrame({
    'Modelo': ['Baseline', 'Regresión logística'],
    'Brier score': [brier_base, brier_log],
    'Log loss': [logloss_base, logloss_log]
})

display(resultados_val.round(6))


El modelo obtuvo un **log loss menor** que el baseline (0.072498 frente a 0.073692), pero un **Brier score ligeramente mayor** (0.013885 frente a 0.013754). Por tanto, en esta validación no encontramos una mejora uniforme de la calidad probabilística.

### A.6. Capacidad de discriminación

Para estudiar si el modelo ordenaba los incidentes fatales por encima de los no fatales calculamos **Average Precision (AP)**. En el baseline constante, esta métrica coincide con la prevalencia de fatalidad del periodo evaluado.

In [ ]:
from sklearn.metrics import average_precision_score

ap_base = average_precision_score(y_val, proba_base_val)
ap_log = average_precision_score(y_val, proba_log_val)

print(f'AP baseline:            {ap_base:.4f}')
print(f'AP regresión logística: {ap_log:.4f}')

La AP aumentó de **0.0139 a 0.0356**, alrededor de **2.56 veces** el valor del baseline. Las variables seleccionadas aportaron cierta capacidad de discriminación, aunque esta mejora no resolvió la diferencia observada en Brier score.

### A.7. Calibración inicial

Finalmente comparamos las probabilidades de la regresión logística con las frecuencias observadas. Agrupamos las predicciones en seis intervalos de tamaño similar para construir una curva de calibración sobre julio–agosto. Esta comparación es descriptiva; no ajustamos un calibrador adicional.

In [ ]:
import matplotlib.pyplot as plt
from sklearn.calibration import calibration_curve

frac_pos, mean_pred = calibration_curve(
    y_val, proba_log_val, n_bins=6, strategy='quantile'
)

fig, ax = plt.subplots(figsize=(8, 6))
ax.plot(mean_pred, frac_pos, marker='o', label='Regresión logística')
limite = max(mean_pred.max(), frac_pos.max()) * 1.1
ax.plot([0, limite], [0, limite], '--', color='gray', label='Calibración ideal')
ax.set(xlabel='Probabilidad estimada', ylabel='Frecuencia observada de fatalidad',
       title='Calibración inicial — Regresión logística')
ax.legend()
ax.grid(alpha=0.2)
plt.show()


La curva quedó cerca de la diagonal ideal en varios grupos de probabilidades bajas. En el grupo de predicciones más altas, el promedio estimado fue cercano al **5.4%**, frente a una frecuencia observada aproximada del **2.3%**. Esta desviación sugiere una sobreestimación en ese segmento, aunque la validación incluyó solamente **66 incidentes fatales**.

### A.8. Cierre del experimento

La regresión logística mostró señales de discriminación, pero los resultados de validación no respaldaron todavía la calidad de probabilidad que necesitaríamos para consultas geográficas pequeñas. Por ello detuvimos este experimento en lugar de aumentar la complejidad de los modelos sin resolver primero la pregunta de utilidad del Atlas.

Conservamos los resultados como punto de partida para un estudio posterior si es que en el futuro disponemos de registros comparables de otros años. En el cuerpo principal continuaremos con el análisis espacio-temporal y la composición vehicular de los incidentes registrados.